# Generative Morphology (AE + Flow) on Euclid VIS 64px galaxies

End-to-end walkthrough of SHINE's learned-morphology tier on real Euclid VIS
data, from the raw quadrant image to a MAP fit of the observed galaxies:

1. **The data** — what a Euclid VIS quadrant frame actually contains
   (science / RMS / flag / background maps, PSF grid, MER catalogue).
2. **Source selection** — only the galaxies kept by the SHINE cuts *and*
   small enough to live on the 64x64 stamp tier (the tier the generative
   model was trained for); their cutouts are displayed.
3. **The generative model** — the frozen AutoEncoder + normalizing flow
   (`shine.morphology`), loaded from checkpoints.
4. **10 galaxies without PSF** — `z ~ flow`, `g = AE.decode(z)`.
5. **The same 10 galaxies with residual PSFs** — the residual-PSF grid
   `PSF_3-4-F_residual.fits.gz` bilinearly interpolated at 10 detector
   positions (the PSFs used are displayed too).
6. **Per-galaxy MAP, no shear** — the first thing worth checking: can the
   decoder reproduce each observed galaxy at all? Each galaxy is fitted on
   its own 64x64 stamp with `g1 = g2 = 0` fixed.
7. **Joint MAP with shear** — only then, the full scene fitted with
   `MultiExposureScene` + `Inference.run_map`.
8. **Results** — model / residual images, per-galaxy stamps, inferred shear.

> **Why the *residual* PSF and not the full local PSF?** The AE was trained
> with `decode(z)` convolved by `psf_residual`, i.e. the kernel relating the
> true local PSF to a fixed isotropic reference PSF (`PSFiso`). So
> `decode(z) ~= G_true (*) PSFiso`: the reference PSF is still baked into the
> decoder output. Convolving it with the *full* local PSF would apply
> `PSFiso` twice and over-blur the stamp. See
> `shine/morphology/psf_residual.py` and `data/LEARNED_MORPHOLOGY_NOTES.md`.

## 0. Setup (Colab, GPU T4)

This notebook is meant to run on Colab: `flowjax` + `JAX-GalSim` + the
pinned `equinox` are awkward to install alongside a local JAX. Run the two
cells below on a fresh runtime, then restart the runtime if Colab asks.

**Pick a GPU runtime** (*Runtime → Change runtime type → T4 GPU*) before
running anything. Everything here works on CPU, but the two MAP fits
(sections 6 and 7) re-render every galaxy through the AE decoder plus
JAX-GalSim FFTs at every optimisation step — on 64x64 stamps in section 6,
on all three 2048x2066 exposures in section 7 — which is what makes CPU
painful; a T4 has ample memory for these settings (the whole
scene is ~50 MB of float32 images). The setup cell below deliberately does
**not** pin `jax`, so Colab's CUDA-enabled build is kept — check that
`jax.devices()` reports a `cuda` device in the next cell; if it prints CPU
after the installs, `jax` was upgraded past its CUDA plugin, so restart the
runtime (or `pip install -q "jax[cuda12]"`) before continuing.

The repo carries both the Euclid test data (`data/EUC_VIS_SWL/`) and the
AE/flow checkpoints (`wandb_weights/`) through **git-lfs**, so `git lfs pull`
is required — without it those files are 130-byte pointer stubs and every
`fits.open` / checkpoint load below fails.

In [ ]:
# Dependencies. Pin equinox (the checkpoints were serialised with 0.13.6) and
# flowjax explicitly -- letting pip resolve the whole stack freely can hit
# "resolution-too-deep" and never finish.
# NB: do NOT pin paramax==0.0.4 -- flowjax 17.2.1 requires paramax>=0.0.5 and
# the pin makes the resolver fail outright.
!pip install -q "equinox==0.13.6" "einops>=0.8,<0.9"
!pip install -q "flowjax==17.2.1" "paramax>=0.0.5"
!pip install -q git+https://github.com/GalSim-developers/JAX-GalSim.git

In [ ]:
# Clone SHINE with its LFS payload (data + AE/flow checkpoints), then install.
!git lfs install
!git clone -b GenGal64 https://github.com/VincentB03/SHINE.git SHINE
%cd SHINE
!git lfs pull
!pip install -q -e .

In [ ]:
import logging
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=".*complex128.*", module="jax_galsim")

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

from shine.config import InferenceConfig, MAPConfig
from shine.euclid.config import (
    EuclidDataConfig,
    EuclidInferenceConfig,
    SourceSelectionConfig,
)
from shine.euclid.data_loader import EuclidDataLoader, EuclidPSFModel
from shine.euclid.plots import plot_exposure_comparison
from shine.euclid.scene import MultiExposureScene, render_model_images
from shine.inference import Inference
from shine.morphology.config import LearnedMorphologyConfig
from shine.morphology.loader import load_frozen_autoencoder, load_frozen_flow
from shine.morphology.render import render_learned_galaxy
import jax_galsim as galsim

%matplotlib inline

# Run from either the repo root or notebooks/.
REPO_ROOT = Path.cwd() if (Path.cwd() / "shine").is_dir() else Path.cwd().parent
DATA_DIR = REPO_ROOT / "data" / "EUC_VIS_SWL"
QUADRANT = "3-4.F"

# Frozen AE + flow checkpoints (committed in the repo, git-lfs).
AE_CHECKPOINT_DIR = str(REPO_ROOT / "wandb_weights" / "i1pf186a" / "epoch_2000")
AE_EPOCH = 2000
FLOW_CHECKPOINT_DIR = str(REPO_ROOT / "wandb_weights" / "4q23te9a" / "epoch_420")
FLOW_EPOCH = 420

# Source selection + inference settings (small on purpose: every source on
# the learned tier costs one AE decode per exposure per optimisation step).
MIN_SNR = 20.0
MAX_SOURCES = 20
MAP_STEPS = 150
LEARNING_RATE = 0.002
RNG_SEED = 42
N_SHOW = 10  # galaxies generated / PSFs displayed / stamps inspected

# The data loader logs the full source-selection cascade at INFO level.
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
logging.getLogger("jax").setLevel(logging.WARNING)

print("Repo root:", REPO_ROOT)
print("Data dir :", DATA_DIR, "(exists:", DATA_DIR.is_dir(), ")")
print("JAX devices:", jax.devices())
if jax.devices()[0].platform != "gpu":
    print("WARNING: running on CPU -- the MAP fits (sections 6-7) will be slow. "
          "Runtime > Change runtime type > T4 GPU.")

# Guard against un-pulled git-lfs pointer stubs (a few hundred bytes instead
# of tens of MB) -- they produce confusing FITS/deserialisation errors later.
for path in [DATA_DIR / "PSF_3-4-F_residual.fits.gz",
             Path(AE_CHECKPOINT_DIR) / f"model_checkpoint_{AE_EPOCH}.eqx"]:
    assert path.exists(), f"missing: {path}"
    assert path.stat().st_size > 10_000, f"{path} looks like a git-lfs pointer -- run `git lfs pull`"

## 1. The data: a Euclid VIS quadrant frame

`data/EUC_VIS_SWL/` holds **real Euclid Q1 VIS data**: observation 2704
(NGC 6505, DEEP mode, 2024-07-18), quadrant **3-4.F** of the VIS focal
plane — the quadrant closest to the pointing centre — extracted from three
overlapping dithered exposures (560.52 s each) via the ESA Euclid Science
Archive.

Per exposure the file carries three 2048x2066 planes:

| Extension | Content |
|-----------|---------|
| `3-4.F.SCI` | Calibrated science image, **in ADU, not background-subtracted** |
| `3-4.F.RMS` | Per-pixel noise sigma (ADU) — the likelihood's weights |
| `3-4.F.FLG` | Data-quality bitmask (bad pixels, cosmic rays, ghosts, star halos) |

plus, as separate products:

- `EUC_VIS_SWL-BKG-*` — pipeline background maps (one per dither);
- `PSF_3-4-F.fits.gz` — the local PSF model as a 9x9 grid of 21x21 stamps
  tiled into one image, bilinearly interpolated at each source position;
- `PSF_3-4-F_residual.fits.gz` — the same grid divided (in Fourier space) by
  the isotropic reference PSF, i.e. the kernel the learned tier needs;
- `catalogue_3-4-F.fits.gz` — the MER catalogue (positions, fluxes, sizes,
  quality flags) driving source selection.

Pixel scale: 0.1"/px. Photometry: `m_AB = -2.5 log10(ADU) + MAGZEROP + 2.5 log10(EXPTIME)`.

**Background subtraction.** The SCI planes are *not* background-subtracted on
disk; SHINE does it at load time in
`EuclidExposure.prepare_image_data`, which is what fills
`ExposureSet.images`:

```python
image = self.sci - background_map      # when background_paths is configured
image = self.sci - sigma_clipped_median(self.sci[mask])   # fallback otherwise
```

So `data.images[j]` below is `SCI - BKG` **pixel by pixel** (not a scalar
level), and the scene model fits that, with no background term of its own —
hence the `background_paths=` argument in the next section: without it the
loader falls back to one sigma-clipped median for the whole quadrant and the
leftover straylight structure gets absorbed into the galaxy models. (The
`EuclidInferenceConfig.background` field — `"fit"`/`"median"`/`"fixed"` — is
declared but not read by any code path today; only `background_paths`
matters.) The same masking step turns `FLG & bad_pixel_mask` pixels into
`sigma = 1e10`, i.e. zero weight in the likelihood.

In [ ]:
exposure_paths = sorted(str(p) for p in DATA_DIR.glob("EUC_VIS_SWL-DET-*_3-4-F.fits.gz"))
bkg_paths = sorted(str(p) for p in DATA_DIR.glob("EUC_VIS_SWL-BKG-*_3-4-F.fits.gz"))
assert len(exposure_paths) == 3 and len(bkg_paths) == 3

# Inspect one raw exposure directly (before any SHINE processing).
with fits.open(exposure_paths[0]) as hdul:
    hdul.info()
    sci_hdu = hdul[f"{QUADRANT}.SCI"]
    sci_raw = sci_hdu.data.astype(np.float32)
    rms_raw = hdul[f"{QUADRANT}.RMS"].data.astype(np.float32)
    flg_raw = hdul[f"{QUADRANT}.FLG"].data.astype(np.int32)
    hdr = sci_hdu.header

with fits.open(bkg_paths[0]) as hdul:
    bkg_raw = hdul[QUADRANT].data.astype(np.float32)

print()
for key in ("EXPTIME", "GAIN", "RDNOISE", "MAGZEROP", "CTYPE1", "CTYPE2"):
    print(f"  {key:9s} = {hdr.get(key)}")
print(f"  shape     = {sci_raw.shape}")
print(f"  SCI range = [{sci_raw.min():.1f}, {sci_raw.max():.1f}] ADU")
print(f"  flagged   = {(flg_raw != 0).mean() * 100:.2f}% of pixels")

In [ ]:
# Displayed with an arcsinh stretch: a linear scale is dominated by the few
# bright stars and shows nothing of the faint galaxies we actually model.
# Second panel = exactly what prepare_image_data will hand to the model:
# the science plane minus the per-pixel background map.
panels = [
    (np.arcsinh(sci_raw), "SCI, raw [arcsinh(ADU)]", "gray_r"),
    (np.arcsinh(sci_raw - bkg_raw), "SCI - BKG map, per pixel [arcsinh(ADU)]", "gray_r"),
    (bkg_raw, "Background map [ADU]", "viridis"),
    (rms_raw, "RMS noise map [ADU]", "magma"),
    (flg_raw != 0, "Flagged pixels (FLG != 0)", "gray"),
]

fig, axes = plt.subplots(1, len(panels), figsize=(5.5 * len(panels), 6))
for ax, (img, title, cmap) in zip(axes, panels):
    arr = np.asarray(img, dtype=np.float32)
    finite = arr[np.isfinite(arr)]
    vmin, vmax = np.percentile(finite, [1, 99]) if finite.size else (0, 1)
    im = ax.imshow(arr, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("x [px]")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
axes[0].set_ylabel("y [px]")
fig.suptitle(
    f"Euclid VIS quadrant {QUADRANT} — obs 2704 (NGC 6505), dither 0 — "
    f"{sci_raw.shape[1]}x{sci_raw.shape[0]} px @ 0.1\"/px",
    fontsize=13,
)
fig.tight_layout()
plt.show()

## 2. Source selection: the galaxies kept by the 64x64 cuts

`EuclidDataLoader` applies the `SourceSelectionConfig` cuts to the MER
catalogue (SNR, VIS detection, spurious / point-source flags, detection
quality bitmask) and then assigns each surviving source a stamp tier from
`galaxy_stamp_sizes`: the smallest stamp that fits `~3 * hlr` on each side
plus the PSF half-width, i.e. `2 * (3 * hlr_px + 10.5) <= stamp`.

Here `galaxy_stamp_sizes = [64]`, which is exactly the "64x64 cut": sources
too extended for a 64px stamp are dropped by `_select_sources`, so **every**
remaining galaxy sits on the learned tier the AE was trained for
(64x64 px @ 0.1"/px).

In [ ]:
learned_morphology = LearnedMorphologyConfig(
    enabled=True,
    ae_checkpoint_dir=AE_CHECKPOINT_DIR,
    ae_epoch=AE_EPOCH,
    flow_checkpoint_dir=FLOW_CHECKPOINT_DIR,
    flow_epoch=FLOW_EPOCH,
    apply_to_stamp_size=64,
    # NOT PSF_3-4-F.fits.gz -- see the note at the top of the notebook.
    psf_residual_path=str(DATA_DIR / "PSF_3-4-F_residual.fits.gz"),
)

config = EuclidInferenceConfig(
    data=EuclidDataConfig(
        exposure_paths=exposure_paths,
        psf_path=str(DATA_DIR / "PSF_3-4-F.fits.gz"),
        catalog_path=str(DATA_DIR / "catalogue_3-4-F.fits.gz"),
        background_paths=bkg_paths,   # real per-pixel background, not a median
        quadrant=QUADRANT,
    ),
    sources=SourceSelectionConfig(min_snr=MIN_SNR, max_sources=MAX_SOURCES),
    inference=InferenceConfig(
        method="map",
        map_config=MAPConfig(enabled=True, num_steps=MAP_STEPS, learning_rate=LEARNING_RATE),
        rng_seed=RNG_SEED,
    ),
    galaxy_stamp_sizes=[64],          # the 64x64 cut
    learned_morphology=learned_morphology,
)

data = EuclidDataLoader(config).load()

print()
print(f"Sources kept        : {data.n_sources} (all on the 64px learned tier)")
print(f"Exposures           : {data.n_exposures}  |  image {data.image_ny}x{data.image_nx}")
print(f"Catalog flux [ADU]  : [{float(data.catalog_flux_adu.min()):.0f}, {float(data.catalog_flux_adu.max()):.0f}]")
print(f"Catalog HLR [arcsec]: [{float(data.catalog_hlr_arcsec.min()):.3f}, {float(data.catalog_hlr_arcsec.max()):.3f}]")
print(f"Residual PSF stamps : {None if data.psf_residual_images is None else tuple(data.psf_residual_images.shape)}")
assert int(np.asarray(data.source_stamp_tier).max()) == 0, "some source is not on the 64px tier"

# The images the model is fitted to are the per-pixel background-subtracted
# science planes (see the note in section 1), not the raw SCI planes.
np.testing.assert_allclose(np.asarray(data.images[0]), sci_raw - bkg_raw, rtol=0, atol=1e-3)
print(f"\ndata.images[0] == SCI - BKG (per pixel) ✓   "
      f"median = {float(np.median(data.images[0])):.3f} ADU (raw SCI: {np.median(sci_raw):.1f} ADU)")

In [ ]:
pos0 = np.asarray(data.pixel_positions[:, 0, :])   # positions in exposure 0
visible0 = np.asarray(data.source_visible[:, 0])
image0 = np.asarray(data.images[0])                # background-subtracted

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))

axes[0].imshow(np.arcsinh(image0), origin="lower", cmap="gray_r",
               vmin=np.percentile(np.arcsinh(image0), 1),
               vmax=np.percentile(np.arcsinh(image0), 99.5), interpolation="nearest")
axes[0].scatter(pos0[visible0, 0], pos0[visible0, 1], s=60, facecolors="none",
                edgecolors="#F44336", linewidths=1.2, label="visible in exp 0")
axes[0].scatter(pos0[~visible0, 0], pos0[~visible0, 1], s=60, marker="x",
                color="#2196F3", label="outside exp 0")
axes[0].set_xlim(0, data.image_nx)
axes[0].set_ylim(0, data.image_ny)
axes[0].set_title(f"Selected galaxies (N={data.n_sources})")
axes[0].set_xlabel("x [px]"); axes[0].set_ylabel("y [px]")
axes[0].legend(fontsize=8, loc="upper right")

axes[1].hist(np.asarray(data.catalog_hlr_arcsec), bins=15, color="#4CAF50", edgecolor="k")
axes[1].set_xlabel("catalog HLR [arcsec]"); axes[1].set_ylabel("count")
axes[1].set_title("Size distribution (all fit in 64 px)")

axes[2].hist(np.log10(np.asarray(data.catalog_flux_adu)), bins=15, color="#FF9800", edgecolor="k")
axes[2].set_xlabel("log10(catalog flux [ADU])"); axes[2].set_ylabel("count")
axes[2].set_title("Flux distribution")

fig.tight_layout()
plt.show()

In [ ]:
STAMP = 64
half = STAMP // 2


def cutout(image, x, y, size=STAMP):
    # Square cutout centred on a (rounded) pixel position; None at the edge.
    xi, yi = int(round(float(x))), int(round(float(y)))
    y0, y1, x0, x1 = yi - size // 2, yi + size // 2, xi - size // 2, xi + size // 2
    if y0 < 0 or x0 < 0 or y1 > image.shape[0] or x1 > image.shape[1]:
        return None
    return image[y0:y1, x0:x1]


# Galaxies with a complete, non-truncated 64x64 cutout in exposure 0.
gal_indices = [
    i for i in range(data.n_sources)
    if visible0[i] and cutout(image0, *pos0[i]) is not None
]
print(f"{len(gal_indices)} / {data.n_sources} galaxies have a full 64x64 cutout in exposure 0")
assert gal_indices, "no galaxy with a complete cutout -- lower MIN_SNR or raise MAX_SOURCES"

# Sections 5 and 7 inspect these galaxies one by one; never ask for more
# than we actually have.
N_SHOW = min(N_SHOW, len(gal_indices))

n_cols = 5
n_rows = max(1, int(np.ceil(len(gal_indices) / n_cols)))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.4 * n_cols, 2.6 * n_rows))
for ax, i in zip(np.atleast_1d(axes).ravel(), gal_indices):
    stamp = cutout(image0, *pos0[i])
    ax.imshow(np.arcsinh(stamp), origin="lower", cmap="gray_r")
    ax.set_title(f"#{i} | {float(data.catalog_hlr_arcsec[i]):.2f}\"", fontsize=8)
    ax.axis("off")
for ax in np.atleast_1d(axes).ravel()[len(gal_indices):]:
    ax.axis("off")
fig.suptitle("Observed 64x64 cutouts of the selected galaxies (exposure 0, arcsinh)", fontsize=12)
fig.tight_layout()
plt.show()

## 3. The generative model (frozen AutoEncoder + normalizing flow)

`shine.morphology` holds two frozen `equinox` checkpoints:

- the **AutoEncoder**, whose decoder maps a latent code `z` to a 64x64
  galaxy light profile at 0.1"/px (softplus output, so non-negative);
- the **normalizing flow**, a learned prior over those latent codes — this
  is what makes the morphology a *probabilistic* model rather than a fixed
  template, and what `sample_latent_codes` turns into a NumPyro sample site.

Both are loaded in inference mode (dropout disabled), so `decode` is
deterministic, and neither is ever optimised during inference.

In [ ]:
ae = load_frozen_autoencoder(AE_CHECKPOINT_DIR, AE_EPOCH)
flow = load_frozen_flow(FLOW_CHECKPOINT_DIR, FLOW_EPOCH)

latent_flat = int(np.prod(flow.latent_dim))

print(f"AE   : stamp {ae.nx}x{ae.ny} px @ {ae.scale}\"/px")
print(f"Flow : latent_dim={list(flow.latent_dim)} ({latent_flat} values), cond_dim={flow.cond_dim}")

assert ae.nx == ae.ny == 64, "unexpected AE stamp size"
assert abs(ae.scale - config.data.pixel_scale) < 1e-6, "AE scale != data pixel scale"
assert flow.cond_dim is None, "sample_latent_codes only supports unconditional flows"
assert tuple(ae.encode(jnp.zeros((1, ae.nx, ae.ny)), key=None).shape) == tuple(flow.latent_dim), (
    "AE latent shape != flow.latent_dim -- this AE/flow pair was not trained together"
)

## 4. Generating 10 galaxies — no PSF

`z ~ flow`, `g = AE.decode(z)`. These are the intrinsic profiles the model
proposes: no instrument response applied yet (strictly: they still carry the
fixed isotropic reference PSF from training, which is exactly why the next
section convolves with the *residual* PSF and not the full one).

Two things to keep in mind while looking at them, because both make the
samples *look* wrong when they are not:

- **A Euclid galaxy is small.** At 0.1"/px a typical half-light radius of
  0.2-0.4" is 2-4 px, so the galaxy covers a few percent of a 64x64 stamp.
  Displayed full-frame on a linear scale, every sample looks like the same
  tiny dot. The mosaic below therefore shows a zoom on the central 24x24 px
  with an arcsinh stretch, which is where the morphology actually is.
- **The prior is a magnitude-limited population.** The flow learned the
  distribution of `VincentB03/euclid-Q1-VF`, dominated by faint, small,
  barely-resolved galaxies — so most draws *should* be small and similar.
  The galaxies loaded in section 2, by contrast, are the `MAX_SOURCES`
  **brightest** sources of the quadrant (`_select_sources` sorts by SNR
  descending and truncates), i.e. the opposite tail. Comparing "typical
  prior draw" with "brightest observed galaxy" is not a fair comparison;
  the flux histogram below makes that explicit.

The cells also check the two generative paths agree — `flow.sample()` versus
what the inference actually uses, `sample_latent_codes` (`z_base` pushed
through `flow.forward`). They must produce the same distribution; they did
**not** before `shine.morphology.prior` was fixed to use the flow's own
(trained, non-standard) base `loc`/`scale`.

In [ ]:
# --- Diagnostic: do the two generative paths agree? -----------------------
# flow.sample() is the flow's own generative process. sample_latent_codes()
# (used by MultiExposureScene) instead draws z_base and pushes it through
# flow.forward, so it must apply the flow's *trained* base loc/scale --
# flowjax keeps those trainable, and this checkpoint's are not (0, 1).
import paramax

base = paramax.unwrap(flow.flow).base_dist
base_loc, base_scale = jnp.asarray(base.loc), jnp.asarray(base.scale)
print("flow base loc  :", np.round(np.asarray(base_loc), 3))
print("flow base scale:", np.round(np.asarray(base_scale), 3))
if np.abs(np.asarray(base_loc)).max() > 0.05:
    print("  -> NOT a standard normal: pushing N(0,1) through flow.forward "
          "would sample a different prior than the trained one.")

n_check = 4000
ref = np.asarray(flow.sample(key=jax.random.key(11), sample_shape=(n_check,)))
eps = jax.random.normal(jax.random.key(12), (n_check, latent_flat))
via_prior = np.asarray(jax.vmap(flow.forward)(base_loc + base_scale * eps))
naive = np.asarray(jax.vmap(flow.forward)(eps))          # the un-fixed path

for label, z in (("sample_latent_codes path", via_prior), ("naive N(0,1) path", naive)):
    print(f"{label:26s}: max |mean - flow.sample mean| = "
          f"{np.abs(z.mean(0) - ref.mean(0)).max():.3f}, "
          f"|z|>5 (outside the AE's softclip2 range): {(np.abs(z) > 5).mean() * 100:.2f}%")

# --- 10 generated galaxies -------------------------------------------------
z_gen = flow.unflatten_latent(flow.sample(key=jax.random.key(0), sample_shape=(N_SHOW,)))
galaxies_nopsf = jax.vmap(lambda zi: ae.decode(zi, key=None))(z_gen)[:, 0]

print("\nz shape:", z_gen.shape, "| decoded shape:", galaxies_nopsf.shape)
assert jnp.all(jnp.isfinite(galaxies_nopsf)) and jnp.all(galaxies_nopsf >= 0)


def second_moments(img, pixel_scale=0.1):
    # Unweighted second moments -> size sigma [arcsec] and |e|, enough to tell
    # whether the samples really are morphologically alike.
    img = np.clip(np.asarray(img, dtype=np.float64), 0, None)
    total = img.sum()
    if total <= 0:
        return np.nan, np.nan
    ny, nx = img.shape
    y, x = np.mgrid[0:ny, 0:nx]
    xc, yc = (x * img).sum() / total, (y * img).sum() / total
    dx, dy = x - xc, y - yc
    qxx = (img * dx ** 2).sum() / total
    qyy = (img * dy ** 2).sum() / total
    qxy = (img * dx * dy).sum() / total
    sigma = (max(qxx * qyy - qxy ** 2, 0.0)) ** 0.25 * pixel_scale
    denom = qxx + qyy
    e = np.hypot(qxx - qyy, 2 * qxy) / denom if denom > 0 else np.nan
    return sigma, e


zoom = 24  # px, centred: where a Euclid galaxy actually lives in a 64px stamp
lo, hi = (ae.nx - zoom) // 2, (ae.nx + zoom) // 2

fig, axes = plt.subplots(2, 5, figsize=(15, 6.8))
for k, ax in enumerate(axes.ravel()):
    img = np.asarray(galaxies_nopsf[k])
    sigma, e = second_moments(img, config.data.pixel_scale)
    ax.imshow(np.arcsinh(img[lo:hi, lo:hi] / max(img.max() * 1e-3, 1e-8)),
              origin="lower", cmap="inferno")
    ax.set_title(f"z#{k} | flux={img.sum():.0f}\nsigma={sigma:.2f}\" |e|={e:.2f}", fontsize=8)
    ax.axis("off")
fig.suptitle(
    f"10 generated galaxies — AE.decode(z), z ~ flow prior, no PSF "
    f"(central {zoom}x{zoom} px, arcsinh stretch)", fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Is the sample diversity real, or does the prior collapse to one galaxy?
# Measured on a large batch, and compared against the observed selection.
z_many = flow.unflatten_latent(flow.sample(key=jax.random.key(5), sample_shape=(300,)))
gen_many = jax.vmap(lambda zi: ae.decode(zi, key=None))(z_many)[:, 0]

gen_flux = np.asarray(gen_many.sum(axis=(1, 2)))
moments = np.array([second_moments(g, config.data.pixel_scale) for g in np.asarray(gen_many)])
gen_sigma, gen_e = moments[:, 0], moments[:, 1]

print(f"generated flux  [AE units]: median {np.median(gen_flux):.0f}, "
      f"5-95th [{np.percentile(gen_flux, 5):.0f}, {np.percentile(gen_flux, 95):.0f}]")
print(f"generated size  sigma [\"] : median {np.nanmedian(gen_sigma):.3f}, "
      f"5-95th [{np.nanpercentile(gen_sigma, 5):.3f}, {np.nanpercentile(gen_sigma, 95):.3f}]")
print(f"generated |e|             : median {np.nanmedian(gen_e):.3f}, "
      f"5-95th [{np.nanpercentile(gen_e, 5):.3f}, {np.nanpercentile(gen_e, 95):.3f}]")
print(f"pixel-wise std across samples / mean peak: "
      f"{float(jnp.mean(jnp.std(gen_many, axis=0)) / jnp.mean(jnp.max(gen_many, axis=(1, 2)))):.4f} "
      "(a collapsed prior would sit near 0)")

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].hist(np.log10(np.clip(gen_flux, 1e-3, None)), bins=25, color="#9C27B0",
             edgecolor="k", label="generated (flow prior)")
axes[0].hist(np.log10(np.asarray(data.catalog_flux_adu)), bins=15, color="#FF9800",
             edgecolor="k", alpha=0.6, label=f"selected sources (top {data.n_sources} by SNR)")
axes[0].set_xlabel("log10(total flux)"); axes[0].set_ylabel("count")
axes[0].set_title("Flux: prior population vs. the brightest observed"); axes[0].legend(fontsize=8)

axes[1].hist(gen_sigma[np.isfinite(gen_sigma)], bins=25, color="#4CAF50", edgecolor="k")
axes[1].set_xlabel("size sigma [arcsec]"); axes[1].set_title("Generated sizes")

axes[2].hist(gen_e[np.isfinite(gen_e)], bins=25, color="#2196F3", edgecolor="k")
axes[2].set_xlabel("|e| (unweighted moments)"); axes[2].set_title("Generated ellipticities")
fig.tight_layout()
plt.show()

## 5. The same 10 galaxies with residual PSFs interpolated at 10 positions

`PSF_3-4-F_residual.fits.gz` has the same layout as the ordinary PSF product
(a 9x9 grid of 21x21 stamps tiled into one image), so `EuclidPSFModel` reads
it unchanged and `interpolate_at(x, y)` gives the bilinear combination of the
four surrounding grid stamps, renormalised to unit sum.

We interpolate at the detector positions of the 10 galaxies selected above,
display the kernels used (residual and, for comparison, the full local PSF),
then render each generated galaxy through its residual PSF with
`render_learned_galaxy` — the exact function the scene model calls per source.

In [ ]:
with fits.open(config.data.psf_path) as hdul:
    psf_full_model = EuclidPSFModel(hdul[QUADRANT].data.astype(np.float32))
with fits.open(learned_morphology.psf_residual_path) as hdul:
    psf_res_model = EuclidPSFModel(hdul[QUADRANT].data.astype(np.float32))

print("PSF grid:", psf_res_model.grid_ny, "x", psf_res_model.grid_nx,
      "stamps of", psf_res_model.stamp_size, "px")

psf_indices = np.asarray(gal_indices[:N_SHOW])
psf_positions = pos0[psf_indices]                     # (N_SHOW, 2) detector px
psf_residuals = np.stack([psf_res_model.interpolate_at(x, y) for x, y in psf_positions])
psf_fulls = np.stack([psf_full_model.interpolate_at(x, y) for x, y in psf_positions])

# Same interpolation the data loader already ran for these sources.
np.testing.assert_allclose(
    psf_residuals,
    np.asarray(data.psf_residual_images)[psf_indices, 0],
    rtol=1e-5, atol=1e-7,
)
print("interpolate_at matches ExposureSet.psf_residual_images ✓")
print(f"peak  full PSF: {psf_fulls.max(axis=(1, 2)).mean():.3f} | "
      f"residual PSF: {psf_residuals.max(axis=(1, 2)).mean():.3f}  "
      "(the residual is sharper: the isotropic part is already in decode(z))")

In [ ]:
fig, axes = plt.subplots(2, N_SHOW, figsize=(2.0 * N_SHOW, 4.6))
for k in range(N_SHOW):
    x, y = psf_positions[k]
    axes[0, k].imshow(psf_residuals[k], origin="lower", cmap="viridis")
    axes[0, k].set_title(f"({x:.0f}, {y:.0f})", fontsize=8)
    axes[1, k].imshow(psf_fulls[k], origin="lower", cmap="viridis")
    for row in (0, 1):
        axes[row, k].set_xticks([])
        axes[row, k].set_yticks([])
axes[0, 0].set_ylabel("residual PSF", fontsize=9)
axes[1, 0].set_ylabel("full PSF", fontsize=9)
fig.suptitle(
    "Interpolated 21x21 PSF stamps at the 10 galaxy positions — "
    "top: residual PSF (used by the learned tier), bottom: full local PSF",
    fontsize=12,
)
fig.tight_layout()
plt.show()

In [ ]:
gsparams = galsim.GSParams(minimum_fft_size=128, maximum_fft_size=128)

galaxies_psf = jnp.stack([
    render_learned_galaxy(
        z_gen[k],
        jnp.float32(0.0), jnp.float32(0.0),          # no shear applied here
        jnp.asarray(psf_residuals[k]),               # interpolated residual PSF
        jnp.asarray(data.wcs_jacobians[psf_indices[k], 0]),  # real local WCS Jacobian
        jnp.float32(0.0), jnp.float32(0.0),          # no sub-pixel offset
        jnp.array(True),                             # source visible
        ae, ae.nx, config.data.pixel_scale, gsparams,
    )
    for k in range(N_SHOW)
])

assert jnp.all(jnp.isfinite(galaxies_psf))

fig, axes = plt.subplots(3, N_SHOW, figsize=(2.0 * N_SHOW, 6.6))
for k in range(N_SHOW):
    before, after = np.asarray(galaxies_nopsf[k]), np.asarray(galaxies_psf[k])
    axes[0, k].imshow(before, origin="lower", cmap="inferno")
    axes[1, k].imshow(after, origin="lower", cmap="inferno")
    axes[2, k].imshow(after - before, origin="lower", cmap="RdBu_r",
                      vmin=-np.abs(after - before).max(), vmax=np.abs(after - before).max())
    axes[0, k].set_title(f"z#{k}", fontsize=9)
    for row in range(3):
        axes[row, k].set_xticks([])
        axes[row, k].set_yticks([])
for row, label in enumerate(["no PSF", "* residual PSF", "difference"]):
    axes[row, 0].set_ylabel(label, fontsize=9)
fig.suptitle("Generated galaxies before / after convolution with the interpolated residual PSF", fontsize=13)
fig.tight_layout()
plt.show()

print("flux ratio after/before (convolution should roughly conserve flux):")
print(np.round(np.asarray(galaxies_psf.sum(axis=(1, 2)) / galaxies_nopsf.sum(axis=(1, 2))), 3))

## 6. First check: can the model reproduce each galaxy? (per-galaxy MAP, no shear)

Before asking the model for a shear, ask it whether it can represent the
galaxies at all. Here every galaxy is fitted **independently on its own
64x64 stamp**, with the shear held at `g1 = g2 = 0`, so nothing about the
morphology can leak into a shear estimate: if the decoder cannot reproduce
these galaxies, a `g1/g2` from section 7 would just be absorbing that
mismatch.

Free per galaxy (all sampled inside one `numpyro.plate`, so the galaxies are
optimised together but stay statistically independent — no shared parameter
links them):

- `z` — the latent code, drawn from the same flow prior the scene uses
  (`shine.morphology.prior.sample_latent_codes`, i.e. `z_base ~ N(0, 1)`
  pushed through the flow);
- `log_amp` — a per-galaxy flux scale. **This is not cosmetic**: `decode(z)`
  lives in the AE's training flux units, which are known not to match this
  quadrant's ADU convention (Finding 2 in `data/LEARNED_MORPHOLOGY_NOTES.md`).
  Without a free amplitude the fit would fail on units alone and tell us
  nothing about morphology. The fitted amplitudes are also a direct
  measurement of that unknown calibration factor.
- `dx`, `dy` — sub-pixel recentring, initialised at the true sub-pixel
  residual `pos - round(pos)` (the stamp corner is `round(pos) - 32`, the
  same convention `_render_tier` uses to scatter stamps).

The rendering is `render_learned_galaxy` with the interpolated **residual**
PSF, exactly as in the scene model; the likelihood is the same per-pixel
Gaussian, restricted to the stamp.

In [ ]:
import numpyro
import numpyro.distributions as dist
from shine.morphology.prior import sample_latent_codes

PER_GALAXY_MAP_STEPS = 400
PER_GALAXY_LR = 0.01

fit_indices = np.asarray(gal_indices)
n_fit = len(fit_indices)

# Stamps for every galaxy with a complete cutout in exposure 0.
obs_stamps = jnp.stack([jnp.asarray(cutout(image0, *pos0[i])) for i in fit_indices])
sigma_stamps = jnp.stack([jnp.asarray(cutout(np.asarray(data.noise_sigma[0]), *pos0[i]))
                          for i in fit_indices])
psf_stamps = jnp.asarray(np.asarray(data.psf_residual_images)[fit_indices, 0])
wcs_stamps = jnp.asarray(np.asarray(data.wcs_jacobians)[fit_indices, 0])
visible_stamps = jnp.ones(n_fit, dtype=bool)

# True sub-pixel residual of each source w.r.t. its (rounded) stamp centre.
subpix = (pos0[fit_indices] - np.round(pos0[fit_indices])) * config.data.pixel_scale

print(f"Fitting {n_fit} galaxies independently on {obs_stamps.shape[1]}x{obs_stamps.shape[2]} stamps")
print(f"observed stamp flux [ADU]: median {float(jnp.median(obs_stamps.sum(axis=(1, 2)))):.3e}")

In [ ]:
def render_stamps(z, log_amp, dx, dy):
    # One AE-decoded, residual-PSF-convolved, amplitude-scaled stamp per
    # galaxy. No shear: g1 = g2 = 0, fixed.
    def render_one(z_i, dx_i, dy_i, psf_i, wcs_i, vis_i):
        return render_learned_galaxy(
            z_i, jnp.float32(0.0), jnp.float32(0.0),
            psf_i, wcs_i, dx_i, dy_i, vis_i,
            ae, ae.nx, config.data.pixel_scale, gsparams,
        )

    stamps = jax.vmap(render_one)(z, dx, dy, psf_stamps, wcs_stamps, visible_stamps)
    return jnp.exp(log_amp)[:, None, None] * stamps


def per_galaxy_model(observed_data=None, **extra_args):
    with numpyro.plate("galaxies", n_fit):
        z = sample_latent_codes("z", flow, n_fit)
        # Very weak prior on the flux scale: its order of magnitude is exactly
        # what we are trying to measure here.
        log_amp = numpyro.sample("log_amp", dist.Normal(0.0, 10.0))
        dx = numpyro.sample("dx", dist.Normal(0.0, 0.5))   # arcsec (~5 px)
        dy = numpyro.sample("dy", dist.Normal(0.0, 0.5))

    model_stamps = render_stamps(z, log_amp, dx, dy)
    numpyro.sample(
        "obs",
        dist.Normal(model_stamps, sigma_stamps).to_event(3),
        obs=observed_data,
    )


# Initial amplitude: observed stamp flux / decoded flux at z_base = 0.
z_init = flow.unflatten_latent(jax.vmap(flow.forward)(jnp.zeros((n_fit, latent_flat))))
decoded_init = jax.vmap(lambda zi: ae.decode(zi, key=None))(z_init)[:, 0]
amp_init = jnp.clip(obs_stamps.sum(axis=(1, 2)), 1.0) / jnp.clip(decoded_init.sum(axis=(1, 2)), 1e-6)

per_galaxy_init = {
    "z_base": jnp.zeros((n_fit, latent_flat)),
    "log_amp": jnp.log(amp_init),
    "dx": jnp.asarray(subpix[:, 0]),
    "dy": jnp.asarray(subpix[:, 1]),
}
print("initial amplitude (median):", f"{float(jnp.median(amp_init)):.1f}")

t0 = time.time()
per_galaxy_map = Inference(
    per_galaxy_model,
    InferenceConfig(
        method="map",
        map_config=MAPConfig(enabled=True, num_steps=PER_GALAXY_MAP_STEPS,
                             learning_rate=PER_GALAXY_LR),
        rng_seed=RNG_SEED,
    ),
).run_map(
    jax.random.PRNGKey(RNG_SEED),
    observed_data=obs_stamps,
    init_params=per_galaxy_init,
)
print(f"per-galaxy MAP done in {time.time() - t0:.1f} s "
      f"({PER_GALAXY_MAP_STEPS} steps, {n_fit} galaxies)")

In [ ]:
z_fit = flow.unflatten_latent(jax.vmap(flow.forward)(per_galaxy_map["z_base"]))
model_stamps_fit = render_stamps(
    z_fit, per_galaxy_map["log_amp"], per_galaxy_map["dx"], per_galaxy_map["dy"]
)
chi_stamps = (obs_stamps - model_stamps_fit) / sigma_stamps
chi2_per_gal = np.asarray(jnp.mean(chi_stamps ** 2, axis=(1, 2)))
amp_fit = np.asarray(jnp.exp(per_galaxy_map["log_amp"]))

print(f"chi2/pixel per galaxy: median {np.median(chi2_per_gal):.2f}, "
      f"min {chi2_per_gal.min():.2f}, max {chi2_per_gal.max():.2f}")
print(f"fitted amplitude     : median {np.median(amp_fit):.1f}, "
      f"range [{amp_fit.min():.1f}, {amp_fit.max():.1f}]")
print("  -> this is the empirical AE-units -> ADU calibration factor "
      "(Finding 2 in data/LEARNED_MORPHOLOGY_NOTES.md); a tight distribution "
      "means one scalar would fix it, a broad one means it is galaxy-dependent.")
print(f"sub-pixel recentring |d| [px]: median "
      f"{np.median(np.hypot(per_galaxy_map['dx'], per_galaxy_map['dy'])) / config.data.pixel_scale:.2f}")

In [ ]:
order = np.argsort(chi2_per_gal)          # best fits first
show = np.concatenate([order[:N_SHOW // 2], order[-(N_SHOW - N_SHOW // 2):]])

fig, axes = plt.subplots(4, len(show), figsize=(2.0 * len(show), 8.8))
for col, k in enumerate(show):
    obs = np.asarray(obs_stamps[k])
    mod = np.asarray(model_stamps_fit[k])
    chi = np.asarray(chi_stamps[k])
    lim = float(np.percentile(np.abs(chi), 99))
    vmax = float(np.percentile(obs, 99.5))

    axes[0, col].imshow(np.arcsinh(obs), origin="lower", cmap="gray_r")
    axes[1, col].imshow(np.arcsinh(mod), origin="lower", cmap="gray_r")
    axes[2, col].imshow(chi, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
    axes[3, col].imshow(np.asarray(ae.decode(z_fit[k], key=None)[0]), origin="lower", cmap="inferno")
    axes[0, col].set_title(f"#{fit_indices[k]}\nchi2/px={chi2_per_gal[k]:.1f}", fontsize=8)
    for row in range(4):
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
for row, label in enumerate(["observed", "MAP model", "chi", "decode(z_MAP)"]):
    axes[row, 0].set_ylabel(label, fontsize=9)
fig.suptitle(
    f"Per-galaxy MAP, no shear — {N_SHOW // 2} best and "
    f"{N_SHOW - N_SHOW // 2} worst fits by chi2/pixel",
    fontsize=13,
)
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(chi2_per_gal, bins=15, color="#2196F3", edgecolor="k")
axes[0].axvline(1.0, color="k", ls="--", label="chi2/px = 1 (perfect fit)")
axes[0].set_xlabel("chi2 / pixel"); axes[0].set_ylabel("galaxies")
axes[0].set_title("Reconstruction quality"); axes[0].legend(fontsize=8)
axes[1].hist(np.log10(amp_fit), bins=15, color="#FF9800", edgecolor="k")
axes[1].set_xlabel("log10(fitted amplitude)"); axes[1].set_ylabel("galaxies")
axes[1].set_title("AE-units -> ADU scale factor")
fig.tight_layout()
plt.show()

**How to read this.** `chi2/pixel ~ 1` means the model explains the galaxy
down to the noise. Values well above 1 mean the decoder cannot represent
these morphologies at this flux scale — in that case the shear fit below is
premature, and the fix is upstream (matching the AE's input normalisation /
flux calibration, or retraining on cutouts in this convention), not in the
inference. A tight amplitude histogram means a single scalar would reconcile
the two flux conventions; a broad one means the mismatch is galaxy-dependent
and the learned tier needs an explicit flux parameter of its own.

## 7. Joint MAP with shear on the observed galaxies

Only worth reading if section 6 came out with a sensible `chi2/pixel`:
this step adds the one thing that was deliberately fixed there, the
global shear, and lets all three exposures constrain it at once.

`MultiExposureScene` builds a NumPyro model in which every selected
galaxy's morphology is a latent code sampled from the flow prior
(`z_base -> flow.forward -> z`), decoded by the AE, sheared by the global
`(g1, g2)`, convolved with its **residual** PSF, drawn through the local WCS
Jacobian and scatter-added onto each of the three exposures; the likelihood
is per-pixel Gaussian with the RMS map as sigma.

MAP = `AutoDelta` + Adam through `Inference.run_map`. Latent sites being
optimised: `g1`, `g2`, `z_base` (per source), plus `flux`/`hlr`/`e1`/`e2`/
`dx`/`dy`, which are still sampled by the scene but unused on the learned
tier's rendering path.

In [ ]:
scene = MultiExposureScene(config, data)
model = scene.build_model()
assert scene.ae is not None and scene.flow is not None

# Start from a safe, well-behaved point: zero shear, zero latent (the flow's
# base-distribution mean), catalog flux/size for the unused parametric sites.
init_params = {
    "g1": jnp.float32(0.0),
    "g2": jnp.float32(0.0),
    "flux": jnp.asarray(data.catalog_flux_adu),
    "hlr": jnp.asarray(data.catalog_hlr_arcsec),
    "e1": jnp.zeros(data.n_sources),
    "e2": jnp.zeros(data.n_sources),
    "dx": jnp.zeros(data.n_sources),
    "dy": jnp.zeros(data.n_sources),
    "z_base": jnp.zeros((data.n_sources, latent_flat)),
}

engine = Inference(model, config.inference)
t0 = time.time()
idata = engine.run(
    jax.random.PRNGKey(RNG_SEED),
    observed_data=data.images,
    init_params=init_params,
)
print(f"MAP done in {time.time() - t0:.1f} s ({MAP_STEPS} steps, {data.n_sources} sources, "
      f"{data.n_exposures} exposures)")

## 8. Results of the joint fit

In [ ]:
posterior = idata.posterior
map_params = {name: np.squeeze(posterior[name].values) for name in posterior.data_vars}
print("MAP sites:", {k: np.shape(v) for k, v in map_params.items()})

# AutoDelta returns the *sample* sites, so z_base -- push it through the flow
# exactly as sample_latent_codes does inside the model to recover z.
z_base_map = jnp.asarray(map_params["z_base"]).reshape(data.n_sources, latent_flat)
z_map = flow.unflatten_latent(jax.vmap(flow.forward)(z_base_map))

print(f"\nMAP shear: g1 = {float(map_params['g1']):+.5f}   g2 = {float(map_params['g2']):+.5f}")
print(f"|z| MAP: mean={float(jnp.mean(jnp.abs(z_map))):.3f}, max={float(jnp.max(jnp.abs(z_map))):.3f}")

In [ ]:
render_params = dict(map_params)
render_params["z"] = z_map

model_images = render_model_images(
    render_params,
    data,
    pixel_scale=config.data.pixel_scale,
    stamp_sizes=config.galaxy_stamp_sizes,
    ae=scene.ae,
    learned_tier_idx=0,      # galaxy_stamp_sizes=[64] -> the learned tier is tier 0
)
print("Model images:", model_images.shape)

# Broader residual mask: also drop bright-star halos and ghosts, which the
# scene does not model (bit 5 = GHOST, 18 = STARSIGNAL, 19 = SATURATEDSTAR).
RESIDUAL_EXCLUDE_BITS = 0x1 | (1 << 5) | (1 << 18) | (1 << 19)

for j in range(data.n_exposures):
    fig = plot_exposure_comparison(
        observed=data.images[j],
        model=model_images[j],
        noise_sigma=data.noise_sigma[j],
        mask=data.masks[j],
        exposure_idx=j,
        residual_mask=(data.flag_maps[j] & RESIDUAL_EXCLUDE_BITS) == 0,
    )
    plt.show()

In [ ]:
# Per-galaxy view: observed cutout, MAP model, chi residual, and the
# PSF-free decoded profile the model inferred for that galaxy.
model0 = np.asarray(model_images[0])
sigma0 = np.asarray(data.noise_sigma[0])
inferred_profiles = np.asarray(jax.vmap(lambda zi: ae.decode(zi, key=None))(z_map)[:, 0])

show = gal_indices[:N_SHOW]
fig, axes = plt.subplots(4, len(show), figsize=(2.0 * len(show), 8.6))
for col, i in enumerate(show):
    obs = cutout(image0, *pos0[i])
    mod = cutout(model0, *pos0[i])
    sig = cutout(sigma0, *pos0[i])
    chi = (obs - mod) / sig
    lim = float(np.nanpercentile(np.abs(chi), 99))

    axes[0, col].imshow(np.arcsinh(obs), origin="lower", cmap="gray_r")
    axes[1, col].imshow(np.arcsinh(mod), origin="lower", cmap="gray_r")
    axes[2, col].imshow(chi, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
    axes[3, col].imshow(inferred_profiles[i], origin="lower", cmap="inferno")
    axes[0, col].set_title(f"#{i}  chi2/px={np.nanmean(chi ** 2):.1f}", fontsize=8)
    for row in range(4):
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
for row, label in enumerate(["observed", "MAP model", "chi", "decode(z_MAP)"]):
    axes[row, 0].set_ylabel(label, fontsize=9)
fig.suptitle("Per-galaxy MAP fit (exposure 0)", fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Global goodness of fit on the modelled pixels only.
for j in range(data.n_exposures):
    m = np.asarray((data.flag_maps[j] & RESIDUAL_EXCLUDE_BITS) == 0)
    chi = (np.asarray(data.images[j]) - np.asarray(model_images[j])) / np.asarray(data.noise_sigma[j])
    print(f"exposure {j}: chi2/pixel = {np.nanmean(chi[m] ** 2):.3f} over {m.sum()} pixels")

print(f"\nInferred shear (MAP): g1 = {float(map_params['g1']):+.5f}, g2 = {float(map_params['g2']):+.5f}")

### Caveats when reading these results

- **Flux calibration is an open issue.** `decode(z)` is in the flux units of
  the AE's training set (`VincentB03/euclid-Q1-VF`), which do not obviously
  match this quadrant's ADU convention — a gap of ~2-3 orders of magnitude
  was measured (see "Finding 2 — still open" in
  `data/LEARNED_MORPHOLOGY_NOTES.md`). The learned tier has no free flux
  scale to absorb it, so per-galaxy amplitudes — and therefore the MAP shear
  — should not yet be taken as physical. Resolving it (a documented scalar,
  or an explicit flux parameter on the learned tier) is the prerequisite for
  trusting the numbers above.
- **MAP is a point estimate**, not a posterior: no shear uncertainty here.
  Switch `InferenceConfig(method="nuts")` for credible intervals.
- **Cost scales with sources x exposures x steps**: each optimisation step
  decodes every galaxy through the AE once per exposure. Raise `MAX_SOURCES`
  / `MAP_STEPS` only with a GPU runtime.
- **The natural next validation** is self-consistency: render a scene through
  this same model with a known injected `(g1, g2)`, add noise, re-infer, and
  check the recovered shear.